In [ ]:
####################################
#ENVIRONMENT SETUP

In [ ]:
#LIBRARIES
import os, sys

import numpy as np
import math

import matplotlib
# matplotlib.use("Agg") #UNCOMMENT IF PLOTTING WITHIN JUPYTER DOCUMENT
import matplotlib.pyplot as plt

import cartopy.crs as ccrs
import cartopy.feature as cfeature

import xarray as xr

import pickle 

from tqdm import tqdm

In [ ]:
#Importing DirectoryManager Class
sys.path.append(os.path.join("/glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/CodeFiles/","DataAnalysis","MPAS_Model_Data"))
from CLASSES_Directories import DirectoryManager_Class

In [ ]:
DirectoryManager = DirectoryManager_Class()

codeType = os.path.join("DataAnalysis", "MPAS_Model_Data", "InitialFigures")
dataType = "SurfaceVariableAnimations_Structured"

outputDirectory = DirectoryManager.GetOutputDirectory(codeType, dataType)

In [ ]:
#Importing ModelData Class
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis","MPAS_Model_Data"))
from CLASSES_ModelData import StructuredModelData_Class,DataOperator_Class

In [ ]:
RunType = ("TRACER","MOIST","NSSL")
SimulationTime = ("2022-06-30","2022-07-03")
ModelData = StructuredModelData_Class(DirectoryManager.mainDirectory, DirectoryManager.scratchDirectory, RunType, SimulationTime)

In [ ]:
###############
#FUNCTIONS

In [ ]:
def GetCLims(ModelData, varNames):
    """
    Compute global (min, max) for one or more variables across all timesteps.
    Returns:
        dict[varName] = (vmin, vmax)
    """
    # Make sure varNames is a list
    if isinstance(varNames, str):
        varNames = [varNames]

    # Initialize dictionary of min/max values
    climDictionary = {v: [np.inf, -np.inf] for v in varNames}

    # Loop over timesteps
    for t in tqdm(range(len(ModelData.fileList)-40), desc="Processing timesteps"):
        data = ModelData.GetDataTimestep(t, printout=False)
        data_diag = ModelData.GetDataTimestep_diag(t, printout=False)

        for varName in varNames:
            variableSubset, _, _ = GetVariable_Subset(varName, data, data_diag)
            vmin = variableSubset.min().item()
            vmax = variableSubset.max().item()

            climDictionary[varName][0] = min(climDictionary[varName][0], vmin)
            climDictionary[varName][1] = max(climDictionary[varName][1], vmax)

        data.close()
        data_diag.close()

    # Convert lists → tuples for readability
    climDictionary = {k: tuple(v) for k, v in climDictionary.items()}
    return climDictionary

def LoadOrCreateCLims(ModelData, varNames, filePath="climDictionary.pkl"):
    """
    Loads the climDictionary if it exists. Otherwise, computes it using GetCLims,
    saves it to file, and returns it.
    """
    if os.path.exists(filePath):
        print(f"Loading existing climDictionary from {filePath}")
        with open(filePath, "rb") as f:
            climDictionary = pickle.load(f)
    else:
        print("File not found. Computing new climDictionary.")
        climDictionary = GetCLims(ModelData, varNames)
        with open(filePath, "wb") as f:
            pickle.dump(climDictionary, f)
        print(f"Saved climDictionary to {filePath}")
    
    return climDictionary

In [ ]:
#PlotVariable_with_Borders()

# Preload map features once
COAST = cfeature.COASTLINE.with_scale("50m")
BORDERS = cfeature.BORDERS.with_scale("50m")
STATES = cfeature.STATES.with_scale("50m")
LAND = cfeature.LAND.with_scale("50m")
LAKES = cfeature.LAKES.with_scale("50m")

def PlotVariable_with_Borders(variable, varName, lat,lon, multiplier,
                              outputFile=None, save=False, 
                              cmap="viridis", clim=(None,None), 
                              title=None, units=None):
    """
    Plot a uxarray or xarray variable on a map with coastlines, borders, and states,
    using Matplotlib (static PNG output). Works headlessly — no Selenium needed.
    """

    # Create figure
    fig, ax = plt.subplots(
        subplot_kw={'projection': ccrs.PlateCarree()},
        figsize=(9, 5)
    )

    num_levels=20
    levels = multiplier*np.linspace(clim[0],clim[1],num_levels)
    
    matrix = multiplier*variable.data
    # Scatter/contour fill (tricontourf works for unstructured grids)
    im = ax.contourf(
        lon, lat, matrix,
        levels=levels,
        cmap=cmap,
        transform=ccrs.PlateCarree(),
    )

    # Add map features
    ax.add_feature(COAST, linewidth=1)
    ax.add_feature(BORDERS, linewidth=0.8)
    ax.add_feature(STATES, linewidth=0.5)
    ax.add_feature(LAND, facecolor="lightgray", alpha=0.3)
    ax.add_feature(LAKES, edgecolor="k", facecolor="none")

    # Colorbar
    if units is not None:
        label=varName +fr" (${units}$)"
    else: 
        label=varName
    plt.colorbar(im, ax=ax, orientation="vertical", label=label)

    #LABELS
    # Set extent to your data range (forces lat/lon ticks)
    ax.set_extent([lon.min(), lon.max(), lat.min(), lat.max()], crs=ccrs.PlateCarree())
    
    # Add lat/lon ticks with degrees
    ax.set_xticks(np.linspace(lon.min(), lon.max(), 5), crs=ccrs.PlateCarree())
    ax.set_yticks(np.linspace(lat.min(), lat.max(), 5), crs=ccrs.PlateCarree())
    
    # # Format tick labels as degrees
    # lon_formatter = ccrs.LongitudeFormatter()
    # lat_formatter = ccrs.LatitudeFormatter()
    # ax.xaxis.set_major_formatter(lon_formatter)
    # ax.yaxis.set_major_formatter(lat_formatter)
    if title is not None:
        ax.set_title(title)
    ax.set_xlabel("Longitude (°E)")
    ax.set_ylabel("Latitude (°N)")


    # Save or display
    if save:
        plt.savefig(outputFile, dpi=300, bbox_inches="tight")
        plt.close(fig)
        print(f"Saved image: {outputFile}","\n")
        return None
    else:
        return fig

def SplitTimeString(timeString):
    date, time = timeString.split('_')
    time = time.replace('.', ':')
    return date,time

def GetUnits_Specific(varName):
    if "+" in varName:
        varName = varName.split("+")[0].strip()
        
    for d in (ModelData.unitsDictionary,
              ModelData.unitsDictionary_diag,
              ModelData.unitsDictionary_static):
        if varName in d:
            return d[varName]
    return None
                    
# #TESTING
# t=100
# data = ModelData.GetDataTimestep(t)
# data_diag = ModelData.GetDataTimestep_diag(t); print(f"\n")
# #defining variable names
# varNames = [
#     "q2"]
# # running
# variableDictionary = BuildVariableDictionary(varNames,data,data_diag,climDictionary)
# MakePlots(variableDictionary, save=False)

In [ ]:
def GetVariableOutputFile(varName, t, ModelData, outputDirectory):
    folderName = f"{ModelData.region}_{ModelData.case}_{ModelData.mpType}/{varName}"
    timeString = ModelData.timeStrings[t]
    fileName = f"{varName}_{timeString}.png"
    filePath = DirectoryManager.GetOutputFile(outputDirectory, folderName, fileName)
    return filePath
    
def BuildVariableDictionary(varNames, dataSubset,dataSubset_diag,
                            lat,lon,climDictionary):
    variableDictionary = {}
    for varName in varNames:
        # print(f"Adding {varName}")

        # Getting Output File
        outputFilePath = GetVariableOutputFile(varName, t, ModelData, outputDirectory)
        
        # Handle addition of two variables
        if '+' in varName:
            var1, var2 = varName.split('+')
            var1 = var1.strip()
            var2 = var2.strip()
            
            subset1 = DataOperator_Class.GetData_Variable(ModelData, dataSubset,
                                                          dataSubset_diag,dataSubset_static,var1)
            subset2 = DataOperator_Class.GetData_Variable(ModelData, dataSubset,
                                                          dataSubset_diag,dataSubset_static,var2)
            variableSubset = subset1 + subset2

            # Combine CLims from both variables
            if (var1 in climDictionary) and (var2 in climDictionary):
                vmin = min(climDictionary[var1][0], climDictionary[var2][0])
                vmax = max(climDictionary[var1][1], climDictionary[var2][1])
                clim = (vmin, vmax)
        
        else:
            variableSubset = DataOperator_Class.GetData_Variable(ModelData, 
                                                                 dataSubset,dataSubset_diag,dataSubset_static,varName)
            clim = climDictionary[varName]

        #Setting up Units and Multiplier
        units = GetUnits_Specific(varName).replace(" ", r"\ ")
        if varName in ["qv","qc","qi","qr","q2","qfx"]:
            multiplier = 1/1e3
            units = units.replace('kg', 'g', 1)
        else:
            multiplier = 1
        
        # Store in dictionary
        variableDictionary[varName] = {
            "data": variableSubset,
            "lat": lat,
            "lon": lon,
            "units": units,
            "multiplier": multiplier,
            "outputFilePath": outputFilePath,
            "clim": clim
        }
        
    return variableDictionary

def MakePlots(variableDictionary, save=False):
    date, time = SplitTimeString(ModelData.timeStrings[t])
    title = f"{ModelData.region}/{ModelData.case}/{ModelData.mpType} on {date} at {time}"
    
    for varName, contents in variableDictionary.items():
        # print(f"Plotting {varName}")
    
        data = contents["data"]
        lat  = contents["lat"]
        lon  = contents["lon"]
        units = contents["units"]
        multiplier = contents["multiplier"]
        outputFilePath = contents["outputFilePath"]
        clim = contents["clim"]
        
        fig = PlotVariable_with_Borders(data, varName, lat, lon, multiplier, 
                                        outputFilePath, save=save, 
                                        cmap="viridis", clim=clim,
                                        title=title, units=units)

In [ ]:
#################
#RUNNING

In [ ]:
#defining variable names
t=0
varNames = [
    "u10", #"v10", "q2",
    "hfx", "qfx", "lh",
    "rainnc+rainc",
    "refl10cm_1km"
] + (["greenfrac"] if t == 0 else [])

In [ ]:
# climDictionary = GetCLims(ModelData,varNames) #run only once
climDictionary = LoadOrCreateCLims(ModelData, varNames, filePath="climDictionary.pkl")

In [ ]:
#running
num_times = ModelData.Ntime
for count, t in enumerate(tqdm(range(num_times), desc="Processing timesteps")):
    if t % 10 == 0: print(f"Currently working on time {t}/{num_times}","\n")

    #Loading Data
    [dataSubset, dataSubset_diag, dataSubset_static, lat, lon, _, _] = DataOperator_Class.GetData_Subset(ModelData,t)

    if count == 1:
        varNames.remove("greenfrac")
    
    # runningx
    variableDictionary = BuildVariableDictionary(varNames,dataSubset,dataSubset_diag,
                                                 lat,lon,climDictionary)
    MakePlots(variableDictionary, save=True)

In [ ]:
#################
#MAKING ANIMATION

In [ ]:
#Needed Libraries
# from matplotlib.animation import FuncAnimation, PillowWriter
# from PIL import Image

# from moviepy import VideoFileClip, vfx

#Importing AnimationPlotting_Class
sys.path.append(os.path.join("/glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/CodeFiles/","DataAnalysis","MPAS_Model_Data"))
from CLASSES_PlottingModelData import AnimationPlotting_Class

In [ ]:
def GetPlottingFileName(varName,outputDirectory,ModelData):
    plottingFileName = f"{varName}.gif"
    plottingFilePath = DirectoryManager.GetOutputFile(outputDirectory, 
                                                      f"{ModelData.region}_{ModelData.case}_{ModelData.mpType}/{varName}", 
                                                      plottingFileName)
    return plottingFilePath

In [ ]:
# getting ideal fps
fps = AnimationPlotting_Class.CalculateFPS(num_frames=ModelData.Ntime, time_interval_minutes=15, desired_duration_min=1)

In [ ]:
# running animation
for varName in varNames:
    print(f"Working on {varName}","\n")

    # Setting up output file
    plottingFilePath = GetPlottingFileName(varName,outputDirectory,ModelData)
    AnimationPlotting_Class.CreateAnimation(ModelData, DirectoryManager,
                                            outputDirectory, plottingFilePath, GetVariableOutputFile,
                                            varName, start_t=0, end_t=ModelData.Ntime,
                                            fps=2)

In [ ]:
#converting animation to mp4
varNames2 = varNames.copy(); varNames2.remove("greenfrac")
for varName in varNames2:
    input_file = GetPlottingFileName(varName,outputDirectory,ModelData)
    output_file = input_file.replace(".gif", ".mp4")
    AnimationPlotting_Class.convertGIFtoMP4(input_file, output_file,fps=fps)